# Contour Detection in OpenCV

## Introduction

Contours are one of the most important tools in computer vision and image processing. They are essential for object detection, shape analysis, recognition, and many other applications.

**What are Contours?**

Contours are **curves that join all the continuous points along a boundary** that have the same color or intensity. They represent the shape of objects in an image and are extremely useful for:

- **Object Detection and Recognition**: Identifying and classifying objects in images
- **Shape Analysis**: Measuring properties like area, perimeter, centroid, bounding box
- **Image Segmentation**: Separating objects from the background
- **Motion Tracking**: Following objects across video frames
- **Gesture Recognition**: Detecting hand shapes and positions

**Key Concepts:**

1. Contours work best on **binary images** (black and white)
2. Usually applied after edge detection or thresholding
3. The object to detect should be white on a black background
4. OpenCV provides powerful functions: `cv2.findContours()` and `cv2.drawContours()`

## Import Required Libraries

Let's start by importing the necessary libraries for contour detection.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Configure matplotlib for inline display
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

## Understanding `cv2.findContours()`

The `cv2.findContours()` function detects contours in a binary image. It scans the image and finds boundaries between objects and background.

### Syntax:

```python
contours, hierarchy = cv2.findContours(image, mode, method)
```

### Parameters:

1. **`image`**: Source image (8-bit single-channel, **binary image**)
   - Must be binary: use thresholding or edge detection first
   - Object should be white (255) on black background (0)

2. **`mode`**: Contour retrieval mode - how to organize the contours
   - `cv2.RETR_EXTERNAL`: Retrieves only the **extreme outer contours**
   - `cv2.RETR_LIST`: Retrieves **all contours** without hierarchy
   - `cv2.RETR_CCOMP`: Retrieves contours in **2-level hierarchy** (external and holes)
   - `cv2.RETR_TREE`: Retrieves **all contours** with full hierarchy

3. **`method`**: Contour approximation method - how to represent the contour
   - `cv2.CHAIN_APPROX_NONE`: Stores **all boundary points**
   - `cv2.CHAIN_APPROX_SIMPLE`: Compresses contour by **removing redundant points** (e.g., for a line, stores only endpoints)

### Return Values:

1. **`contours`**: A list of contours found
   - Each contour is a NumPy array of (x, y) coordinates of boundary points
   - Format: `[[[x1, y1]], [[x2, y2]], [[x3, y3]], ...]`

2. **`hierarchy`**: Array containing information about image topology
   - Format: `[Next, Previous, First_Child, Parent]` for each contour
   - Useful for understanding nested contours (objects within objects)

## Creating a Sample Binary Image

Let's create a simple binary image with geometric shapes to demonstrate contour detection.

In [ ]:
# Create a blank black image
image = np.zeros((400, 600), dtype=np.uint8)

# Draw different shapes in white
# Rectangle
cv2.rectangle(image, (50, 50), (150, 150), 255, -1)

# Circle
cv2.circle(image, (250, 100), 50, 255, -1)

# Triangle (using polygon)
triangle_pts = np.array([[400, 50], [450, 150], [350, 150]], np.int32)
cv2.fillPoly(image, [triangle_pts], 255)

# Nested rectangle (for hierarchy demonstration)
cv2.rectangle(image, (100, 250), (250, 350), 255, -1)
cv2.rectangle(image, (130, 280), (220, 320), 0, -1)  # Inner black rectangle

# Small squares
cv2.rectangle(image, (350, 250), (400, 300), 255, -1)
cv2.rectangle(image, (450, 250), (500, 300), 255, -1)

# Display the created image
plt.figure(figsize=(10, 6))
plt.imshow(image, cmap='gray')
plt.title('Binary Image with Shapes')
plt.axis('off')
plt.show()

print(f"Image shape: {image.shape}")
print(f"Image dtype: {image.dtype}")
print(f"Unique values: {np.unique(image)}")

## Example 1: Finding Contours with `RETR_EXTERNAL`

Let's find only the external contours of the shapes in our image.

In [ ]:
# Find contours using RETR_EXTERNAL mode
contours, hierarchy = cv2.findContours(image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

print(f"Number of contours found: {len(contours)}")
print(f"Hierarchy shape: {hierarchy.shape}")
print(f"\nHierarchy:\n{hierarchy}")

# Display information about each contour
for i, contour in enumerate(contours):
    area = cv2.contourArea(contour)
    perimeter = cv2.arcLength(contour, True)
    print(f"\nContour {i}:")
    print(f"  Number of points: {len(contour)}")
    print(f"  Area: {area:.2f} pixels")
    print(f"  Perimeter: {perimeter:.2f} pixels")

## Understanding `cv2.drawContours()`

The `cv2.drawContours()` function draws contours on an image. It's used to visualize the detected contours.

### Syntax:

```python
cv2.drawContours(image, contours, contourIdx, color, thickness)
```

### Parameters:

1. **`image`**: Destination image where contours will be drawn
   - Can be grayscale or color image
   - **Modified in-place**, so make a copy if you want to preserve the original

2. **`contours`**: List of contours (output from `findContours()`)
   - Each contour is a NumPy array of points

3. **`contourIdx`**: Index of the contour to draw
   - **Positive value**: Draw that specific contour (e.g., `0` for first contour)
   - **`-1`**: Draw **all contours**

4. **`color`**: Contour color
   - For grayscale: single value (e.g., `255` for white)
   - For BGR: tuple (e.g., `(0, 255, 0)` for green)

5. **`thickness`** (optional): Thickness of the contour line
   - Positive value: Line thickness in pixels
   - **`-1` or `cv2.FILLED`**: Fill the contour
   - Default: `1`

### Additional Optional Parameters:

- **`lineType`**: Type of line (e.g., `cv2.LINE_AA` for anti-aliased)
- **`hierarchy`**: Hierarchy information from `findContours()`
- **`maxLevel`**: Maximum level to draw (useful with hierarchy)

## Example 2: Drawing All Contours

Let's visualize all the detected contours on a color image.

In [ ]:
# Convert grayscale image to BGR for colored contours
image_color = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

# Draw all contours in green with thickness 2
cv2.drawContours(image_color, contours, -1, (0, 255, 0), 2)

# Display
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(image_color, cv2.COLOR_BGR2RGB))
plt.title('All Contours Drawn in Green')
plt.axis('off')
plt.show()

## Example 3: Drawing Individual Contours with Different Colors

Let's draw each contour with a different color to distinguish them.

In [ ]:
# Create a color image
image_multi_color = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

# Define different colors (BGR format)
colors = [
    (255, 0, 0),    # Blue
    (0, 255, 0),    # Green
    (0, 0, 255),    # Red
    (255, 255, 0),  # Cyan
    (255, 0, 255),  # Magenta
    (0, 255, 255),  # Yellow
]

# Draw each contour with a different color
for i, contour in enumerate(contours):
    color = colors[i % len(colors)]
    cv2.drawContours(image_multi_color, contours, i, color, 3)
    
    # Add text label
    M = cv2.moments(contour)
    if M["m00"] != 0:
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
        cv2.putText(image_multi_color, f'C{i}', (cx-10, cy), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

# Display
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(image_multi_color, cv2.COLOR_BGR2RGB))
plt.title('Each Contour with Different Color')
plt.axis('off')
plt.show()

## Comparison of Contour Retrieval Modes

Different retrieval modes affect how contours are organized and which contours are detected, especially when there are nested contours (objects within objects).

In [ ]:
# Compare different retrieval modes
modes = [
    (cv2.RETR_EXTERNAL, "RETR_EXTERNAL: Only outermost contours"),
    (cv2.RETR_LIST, "RETR_LIST: All contours, no hierarchy"),
    (cv2.RETR_CCOMP, "RETR_CCOMP: Two-level hierarchy"),
    (cv2.RETR_TREE, "RETR_TREE: Full hierarchy tree")
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, (mode, title) in enumerate(modes):
    # Find contours with different modes
    contours_mode, hierarchy_mode = cv2.findContours(image, mode, cv2.CHAIN_APPROX_SIMPLE)
    
    # Create color image
    img_display = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    cv2.drawContours(img_display, contours_mode, -1, (0, 255, 0), 2)
    
    # Display
    axes[idx].imshow(cv2.cvtColor(img_display, cv2.COLOR_BGR2RGB))
    axes[idx].set_title(f'{title}\nContours found: {len(contours_mode)}')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

# Show hierarchy details for RETR_TREE
print("\nHierarchy with RETR_TREE:")
contours_tree, hierarchy_tree = cv2.findContours(image, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
print(f"Number of contours: {len(contours_tree)}")
print(f"Hierarchy shape: {hierarchy_tree.shape}")
print(f"\nHierarchy array [Next, Previous, First_Child, Parent]:")
print(hierarchy_tree)

## Comparison of Contour Approximation Methods

The approximation method affects how many points are stored to represent the contour.

- **`CHAIN_APPROX_NONE`**: Stores all points along the contour
- **`CHAIN_APPROX_SIMPLE`**: Stores only the endpoints of straight lines (much more efficient)

In [ ]:
# Compare approximation methods on a single rectangle
rect_image = np.zeros((200, 300), dtype=np.uint8)
cv2.rectangle(rect_image, (50, 50), (250, 150), 255, -1)

# Method 1: CHAIN_APPROX_NONE (all points)
contours_none, _ = cv2.findContours(rect_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

# Method 2: CHAIN_APPROX_SIMPLE (compressed)
contours_simple, _ = cv2.findContours(rect_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Compare
print(f"CHAIN_APPROX_NONE:")
print(f"  Number of points: {len(contours_none[0])}")
print(f"  First 10 points:\n{contours_none[0][:10].reshape(-1, 2)}")

print(f"\nCHAIN_APPROX_SIMPLE:")
print(f"  Number of points: {len(contours_simple[0])}")
print(f"  All points:\n{contours_simple[0].reshape(-1, 2)}")

print(f"\nCompression ratio: {len(contours_none[0]) / len(contours_simple[0]):.1f}x")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CHAIN_APPROX_NONE
img1 = cv2.cvtColor(rect_image, cv2.COLOR_GRAY2BGR)
cv2.drawContours(img1, contours_none, -1, (0, 255, 0), 2)
for point in contours_none[0][::10]:  # Draw every 10th point
    cv2.circle(img1, tuple(point[0]), 3, (0, 0, 255), -1)
axes[0].imshow(cv2.cvtColor(img1, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'CHAIN_APPROX_NONE\n{len(contours_none[0])} points total')
axes[0].axis('off')

# CHAIN_APPROX_SIMPLE
img2 = cv2.cvtColor(rect_image, cv2.COLOR_GRAY2BGR)
cv2.drawContours(img2, contours_simple, -1, (0, 255, 0), 2)
for point in contours_simple[0]:  # Draw all points
    cv2.circle(img2, tuple(point[0]), 5, (0, 0, 255), -1)
axes[1].imshow(cv2.cvtColor(img2, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'CHAIN_APPROX_SIMPLE\n{len(contours_simple[0])} points (corners only)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Practical Application: Object Detection and Analysis

Let's use contours for a practical task: detecting and analyzing multiple objects in an image.

In [ ]:
# Create a more complex scene with multiple objects
complex_image = np.zeros((400, 600), dtype=np.uint8)

# Add various shapes
shapes = [
    ("Rectangle", (50, 50, 100, 100)),
    ("Circle", (200, 100, 40)),
    ("Ellipse", (350, 100, 60, 40)),
    ("Small Circle", (500, 80, 25)),
    ("Large Rectangle", (100, 250, 180, 120)),
    ("Triangle", [(400, 220), (500, 220), (450, 320)]),
]

# Draw shapes
cv2.rectangle(complex_image, (50, 50), (150, 150), 255, -1)
cv2.circle(complex_image, (200, 100), 40, 255, -1)
cv2.ellipse(complex_image, (350, 100), (60, 40), 0, 0, 360, 255, -1)
cv2.circle(complex_image, (500, 80), 25, 255, -1)
cv2.rectangle(complex_image, (100, 250), (280, 370), 255, -1)
triangle_pts = np.array([[400, 220], [500, 220], [450, 320]], np.int32)
cv2.fillPoly(complex_image, [triangle_pts], 255)

# Find contours
contours_obj, _ = cv2.findContours(complex_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Create output image
output = cv2.cvtColor(complex_image, cv2.COLOR_GRAY2BGR)

# Analyze and annotate each object
print("Object Analysis:")
print("=" * 70)

for i, contour in enumerate(contours_obj):
    # Calculate properties
    area = cv2.contourArea(contour)
    perimeter = cv2.arcLength(contour, True)
    
    # Get bounding box
    x, y, w, h = cv2.boundingRect(contour)
    
    # Get moments for centroid
    M = cv2.moments(contour)
    if M["m00"] != 0:
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
    else:
        cx, cy = 0, 0
    
    # Approximate contour to detect shape
    epsilon = 0.04 * perimeter
    approx = cv2.approxPolyDP(contour, epsilon, True)
    vertices = len(approx)
    
    # Draw contour and bounding box
    color = tuple(np.random.randint(100, 255, 3).tolist())
    cv2.drawContours(output, contours_obj, i, color, 2)
    cv2.rectangle(output, (x, y), (x+w, y+h), (255, 0, 0), 1)
    
    # Draw centroid
    cv2.circle(output, (cx, cy), 4, (0, 255, 255), -1)
    
    # Add label
    cv2.putText(output, f'#{i}', (cx-10, cy-10), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    
    # Print analysis
    print(f"\nObject #{i}:")
    print(f"  Area: {area:.0f} pixels²")
    print(f"  Perimeter: {perimeter:.1f} pixels")
    print(f"  Centroid: ({cx}, {cy})")
    print(f"  Bounding Box: ({x}, {y}, {w}, {h})")
    print(f"  Vertices (approx): {vertices}")
    print(f"  Aspect Ratio: {w/h:.2f}" if h > 0 else "  Aspect Ratio: N/A")

# Display
plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(output, cv2.COLOR_BGR2RGB))
plt.title('Object Detection and Analysis with Contours')
plt.axis('off')
plt.show()

## Working with Real Images: Edge Detection + Contours

In real-world applications, we often need to preprocess images before contour detection. A common pipeline is:

1. **Load image** → Convert to grayscale
2. **Apply blur** → Reduce noise
3. **Edge detection** (Canny) or **Thresholding**
4. **Find contours**
5. **Draw and analyze contours**

In [ ]:
# Create a synthetic "real-world" image with noise
real_image = np.zeros((400, 600), dtype=np.uint8)

# Add shapes
cv2.rectangle(real_image, (100, 100), (200, 200), 200, -1)
cv2.circle(real_image, (350, 150), 60, 220, -1)
cv2.rectangle(real_image, (150, 250), (400, 350), 180, -1)

# Add Gaussian noise to simulate real-world conditions
noise = np.random.normal(0, 25, real_image.shape).astype(np.uint8)
noisy_image = cv2.add(real_image, noise)

# Processing pipeline
# Step 1: Blur to reduce noise
blurred = cv2.GaussianBlur(noisy_image, (5, 5), 0)

# Step 2: Apply Canny edge detection
edges = cv2.Canny(blurred, 50, 150)

# Step 3: Find contours on edge-detected image
contours_edges, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Alternative: Using threshold instead of Canny
_, thresh = cv2.threshold(blurred, 127, 255, cv2.THRESH_BINARY)
contours_thresh, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Visualize the pipeline
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Row 1: Edge detection approach
axes[0, 0].imshow(noisy_image, cmap='gray')
axes[0, 0].set_title('1. Original (with noise)')
axes[0, 0].axis('off')

axes[0, 1].imshow(edges, cmap='gray')
axes[0, 1].set_title('2. Canny Edge Detection')
axes[0, 1].axis('off')

result_edges = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
cv2.drawContours(result_edges, contours_edges, -1, (0, 255, 0), 2)
axes[0, 2].imshow(cv2.cvtColor(result_edges, cv2.COLOR_BGR2RGB))
axes[0, 2].set_title(f'3. Contours from Edges\n({len(contours_edges)} contours)')
axes[0, 2].axis('off')

# Row 2: Threshold approach
axes[1, 0].imshow(noisy_image, cmap='gray')
axes[1, 0].set_title('1. Original (with noise)')
axes[1, 0].axis('off')

axes[1, 1].imshow(thresh, cmap='gray')
axes[1, 1].set_title('2. Binary Threshold')
axes[1, 1].axis('off')

result_thresh = cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)
cv2.drawContours(result_thresh, contours_thresh, -1, (0, 255, 0), 2)
axes[1, 2].imshow(cv2.cvtColor(result_thresh, cv2.COLOR_BGR2RGB))
axes[1, 2].set_title(f'3. Contours from Threshold\n({len(contours_thresh)} contours)')
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

print(f"Edge detection approach found: {len(contours_edges)} contours")
print(f"Thresholding approach found: {len(contours_thresh)} contours")

## Filtering Contours by Properties

Often, we want to filter contours based on properties like area, perimeter, or shape to focus on specific objects.

In [ ]:
# Create image with objects of various sizes
filter_image = np.zeros((400, 600), dtype=np.uint8)

# Add shapes of different sizes
cv2.rectangle(filter_image, (50, 50), (100, 100), 255, -1)      # Small
cv2.rectangle(filter_image, (150, 50), (280, 180), 255, -1)     # Medium
cv2.circle(filter_image, (400, 100), 20, 255, -1)               # Tiny
cv2.circle(filter_image, (500, 100), 60, 255, -1)               # Medium
cv2.rectangle(filter_image, (100, 250), (350, 370), 255, -1)    # Large
cv2.circle(filter_image, (500, 300), 15, 255, -1)               # Tiny

# Find all contours
contours_all, _ = cv2.findContours(filter_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Filter contours by area
min_area = 1000   # Minimum area threshold
max_area = 20000  # Maximum area threshold

large_contours = []
small_contours = []
medium_contours = []

for contour in contours_all:
    area = cv2.contourArea(contour)
    if area < min_area:
        small_contours.append(contour)
    elif area > max_area:
        large_contours.append(contour)
    else:
        medium_contours.append(contour)

# Visualize filtering results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# All contours
img_all = cv2.cvtColor(filter_image, cv2.COLOR_GRAY2BGR)
cv2.drawContours(img_all, contours_all, -1, (0, 255, 0), 2)
axes[0, 0].imshow(cv2.cvtColor(img_all, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title(f'All Contours ({len(contours_all)} objects)')
axes[0, 0].axis('off')

# Small contours
img_small = cv2.cvtColor(filter_image, cv2.COLOR_GRAY2BGR)
cv2.drawContours(img_small, small_contours, -1, (0, 0, 255), 2)
axes[0, 1].imshow(cv2.cvtColor(img_small, cv2.COLOR_BGR2RGB))
axes[0, 1].set_title(f'Small Objects (area < {min_area})\n{len(small_contours)} objects')
axes[0, 1].axis('off')

# Medium contours
img_medium = cv2.cvtColor(filter_image, cv2.COLOR_GRAY2BGR)
cv2.drawContours(img_medium, medium_contours, -1, (255, 255, 0), 2)
axes[1, 0].imshow(cv2.cvtColor(img_medium, cv2.COLOR_BGR2RGB))
axes[1, 0].set_title(f'Medium Objects ({min_area} ≤ area ≤ {max_area})\n{len(medium_contours)} objects')
axes[1, 0].axis('off')

# Large contours
img_large = cv2.cvtColor(filter_image, cv2.COLOR_GRAY2BGR)
cv2.drawContours(img_large, large_contours, -1, (255, 0, 255), 2)
axes[1, 1].imshow(cv2.cvtColor(img_large, cv2.COLOR_BGR2RGB))
axes[1, 1].set_title(f'Large Objects (area > {max_area})\n{len(large_contours)} objects')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

# Print statistics
print("Contour Filtering Results:")
print("=" * 50)
for i, contour in enumerate(contours_all):
    area = cv2.contourArea(contour)
    category = "Small" if area < min_area else ("Large" if area > max_area else "Medium")
    print(f"Contour {i}: Area = {area:.0f} px² → {category}")

## Advanced Drawing Options

The `drawContours()` function has additional parameters for more control over visualization.

In [ ]:
# Create a test image
draw_image = np.zeros((300, 400), dtype=np.uint8)
cv2.rectangle(draw_image, (50, 50), (150, 150), 255, -1)
cv2.circle(draw_image, (250, 100), 50, 255, -1)
cv2.rectangle(draw_image, (150, 180), (300, 250), 255, -1)

# Find contours
contours_draw, _ = cv2.findContours(draw_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Different drawing styles
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 1. Outline only (thin)
img1 = cv2.cvtColor(draw_image, cv2.COLOR_GRAY2BGR)
cv2.drawContours(img1, contours_draw, -1, (0, 255, 0), 1)
axes[0, 0].imshow(cv2.cvtColor(img1, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title('Thin Outline\nthickness=1')
axes[0, 0].axis('off')

# 2. Thick outline
img2 = cv2.cvtColor(draw_image, cv2.COLOR_GRAY2BGR)
cv2.drawContours(img2, contours_draw, -1, (0, 255, 0), 5)
axes[0, 1].imshow(cv2.cvtColor(img2, cv2.COLOR_BGR2RGB))
axes[0, 1].set_title('Thick Outline\nthickness=5')
axes[0, 1].axis('off')

# 3. Filled contours
img3 = np.zeros((300, 400, 3), dtype=np.uint8)
cv2.drawContours(img3, contours_draw, -1, (0, 255, 0), -1)
axes[0, 2].imshow(cv2.cvtColor(img3, cv2.COLOR_BGR2RGB))
axes[0, 2].set_title('Filled Contours\nthickness=-1')
axes[0, 2].axis('off')

# 4. Anti-aliased lines
img4 = cv2.cvtColor(draw_image, cv2.COLOR_GRAY2BGR)
cv2.drawContours(img4, contours_draw, -1, (0, 255, 0), 2, cv2.LINE_AA)
axes[1, 0].imshow(cv2.cvtColor(img4, cv2.COLOR_BGR2RGB))
axes[1, 0].set_title('Anti-aliased\nlineType=LINE_AA')
axes[1, 0].axis('off')

# 5. Draw specific contour only
img5 = cv2.cvtColor(draw_image, cv2.COLOR_GRAY2BGR)
cv2.drawContours(img5, contours_draw, 0, (255, 0, 0), 3)  # Draw first contour in red
axes[1, 1].imshow(cv2.cvtColor(img5, cv2.COLOR_BGR2RGB))
axes[1, 1].set_title('Specific Contour\ncontourIdx=0')
axes[1, 1].axis('off')

# 6. Semi-transparent overlay
img6 = cv2.cvtColor(draw_image, cv2.COLOR_GRAY2BGR)
overlay = img6.copy()
cv2.drawContours(overlay, contours_draw, -1, (0, 255, 0), -1)
cv2.addWeighted(overlay, 0.5, img6, 0.5, 0, img6)
axes[1, 2].imshow(cv2.cvtColor(img6, cv2.COLOR_BGR2RGB))
axes[1, 2].set_title('Semi-transparent Fill\nalpha=0.5')
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

## Summary and Key Takeaways

### `cv2.findContours()` - Finding Contours

**Syntax:**
```python
contours, hierarchy = cv2.findContours(image, mode, method)
```

**Key Points:**
- Requires **binary image** (black background, white objects)
- Returns list of contours and hierarchy information
- Each contour is a NumPy array of (x, y) coordinates

**Retrieval Modes:**
| Mode | Description | Use Case |
|------|-------------|----------|
| `RETR_EXTERNAL` | Only outermost contours | Simple object detection |
| `RETR_LIST` | All contours, no hierarchy | When hierarchy doesn't matter |
| `RETR_CCOMP` | 2-level hierarchy | Detect objects with holes |
| `RETR_TREE` | Full hierarchy tree | Complex nested structures |

**Approximation Methods:**
| Method | Description | Use Case |
|--------|-------------|----------|
| `CHAIN_APPROX_NONE` | Store all points | Need complete boundary |
| `CHAIN_APPROX_SIMPLE` | Store only endpoints | Memory efficiency (recommended) |

---

### `cv2.drawContours()` - Drawing Contours

**Syntax:**
```python
cv2.drawContours(image, contours, contourIdx, color, thickness)
```

**Key Points:**
- Modifies image **in-place** (make a copy if needed)
- Can draw all contours or specific ones
- Supports filling and outlining

**Important Parameters:**
- `contourIdx=-1`: Draw all contours
- `contourIdx=i`: Draw specific contour at index i
- `thickness=-1`: Fill contour
- `thickness>0`: Outline with specified thickness

---

### Common Workflow

```python
# 1. Preprocess image
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray, (5, 5), 0)

# 2. Create binary image
_, binary = cv2.threshold(blurred, 127, 255, cv2.THRESH_BINARY)
# OR
edges = cv2.Canny(blurred, 50, 150)

# 3. Find contours
contours, hierarchy = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# 4. Filter contours (optional)
filtered = [c for c in contours if cv2.contourArea(c) > 100]

# 5. Draw contours
cv2.drawContours(image, filtered, -1, (0, 255, 0), 2)
```

---

### Useful Contour Functions

- `cv2.contourArea(contour)` - Calculate area
- `cv2.arcLength(contour, closed)` - Calculate perimeter
- `cv2.boundingRect(contour)` - Get bounding rectangle
- `cv2.minEnclosingCircle(contour)` - Get minimum enclosing circle
- `cv2.approxPolyDP(contour, epsilon, closed)` - Approximate contour shape
- `cv2.moments(contour)` - Calculate moments (for centroid)

---

### Applications

✅ **Object Detection** - Identify and count objects  
✅ **Shape Analysis** - Classify objects by shape  
✅ **Size Measurement** - Calculate area, perimeter  
✅ **Position Tracking** - Find object centroids  
✅ **Image Segmentation** - Separate foreground/background  
✅ **Quality Control** - Detect defects in manufacturing  
✅ **Gesture Recognition** - Detect hand shapes  
✅ **Document Analysis** - Find text regions

## Practice Exercises

Try these exercises to reinforce your understanding:

### Exercise 1: Shape Classifier
Create a program that:
1. Detects contours in an image
2. Classifies each shape as circle, rectangle, or triangle based on vertices
3. Labels each shape on the image

**Hint:** Use `cv2.approxPolyDP()` to count vertices

---

### Exercise 2: Size Filter
Create a program that:
1. Finds all contours in an image
2. Filters out contours smaller than a certain area
3. Displays only the filtered contours

---

### Exercise 3: Nested Contours
Create an image with shapes inside shapes and:
1. Use `RETR_TREE` mode to find all contours
2. Analyze the hierarchy
3. Draw parent contours in one color and child contours in another

---

### Exercise 4: Real Image Processing
Using a real image or photo:
1. Convert to grayscale
2. Apply Gaussian blur
3. Use Canny edge detection
4. Find and draw contours
5. Count the number of objects detected

---

## Common Pitfalls and Tips

⚠️ **Pitfall 1:** Forgetting to convert to binary image
- **Solution:** Always apply thresholding or edge detection first

⚠️ **Pitfall 2:** Objects not white on black background
- **Solution:** Invert the image if needed: `cv2.bitwise_not(image)`

⚠️ **Pitfall 3:** Too many small contours detected
- **Solution:** Filter by area: `[c for c in contours if cv2.contourArea(c) > min_area]`

⚠️ **Pitfall 4:** Contours not detected properly
- **Solution:** Adjust preprocessing (blur, threshold values, edge detection parameters)

💡 **Tip 1:** Always make a copy before drawing if you need the original
```python
display_image = original_image.copy()
```

💡 **Tip 2:** Use morphological operations to clean up binary images
```python
kernel = np.ones((5,5), np.uint8)
closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
```

💡 **Tip 3:** Combine contour analysis with other OpenCV functions for powerful applications
- Bounding rectangles, rotated rectangles
- Convex hulls
- Shape matching

## Additional Resources

### OpenCV Documentation
- [Contours: Getting Started](https://docs.opencv.org/4.x/d4/d73/tutorial_py_contours_begin.html)
- [Contour Features](https://docs.opencv.org/4.x/dd/d49/tutorial_py_contour_features.html)
- [Contour Properties](https://docs.opencv.org/4.x/d1/d32/tutorial_py_contour_properties.html)

### Related Topics to Explore
- **Morphological Operations** - Clean up binary images before contour detection
- **Convex Hulls** - Find the convex boundary of contours
- **Shape Matching** - Compare contours with `cv2.matchShapes()`
- **Contour Moments** - Calculate centroids, orientation, and other geometric properties
- **Douglas-Peucker Algorithm** - The algorithm behind `cv2.approxPolyDP()`

---

**End of Notebook**

*This notebook covered the fundamentals of contour detection using OpenCV's `findContours()` and `drawContours()` functions. Practice with different images to master these concepts!*